# 02 — Representative Frame Selection

Selects the frame that best represents the ensemble, by combining the
**presence of characteristic hydrogen bond pairs** with **structural similarity
to the average conformation**.

### Method
Each frame is scored on the characteristic hydrogen bond pairs — those
occurring in ≥ `MIN_FREQ` % of frames — weighted by their trajectory-wide
frequency, combined with the Cα RMSD to the average conformation. The frame
maximizing the frequency-weighted identity score while minimizing structural
deviation, across all replicates, is selected as representative.

Concretely, for each frame *i* after the equilibration skip:

```
characteristic = { pair : freq  |  freq >= MIN_FREQ }        # freq in %

identity_i  = sum(freq of characteristic pairs present in frame i)
              / sum(freq of all characteristic pairs)        # 0 - 1

rmsd_norm_i = (rmsd_i - min) / (max - min)                   # 0 - 1, min-max
                                                             # over pooled
                                                             # scored frames

score_i     = IDENTITY_WEIGHT * (1 - identity_i)
            + RMSD_WEIGHT * rmsd_norm_i
```

The lowest-scoring frame wins. `IDENTITY_WEIGHT` and `RMSD_WEIGHT` set the
relative weight of the two terms.

### Input
The pickle written by `01_hbond_analysis.ipynb`, which supplies per replicate
the per-frame residue-pair sets (`frame_pairs`), the per-frame Cα RMSD
(`rmsd_values`), and the file paths (`config`) needed to write the frame out as
a PDB.

`frame_pairs` contains **both** chemical directions, keyed
`(A/B residue, C residue)`. Scoring is therefore direction-agnostic: a pair
counts as present in a frame if it is hydrogen bonded either way.

### Outputs
- Printed summary of the winning frame, the pairs it does and does not contain,
  and the next-best alternatives.
- `<name>_rep_<Rep>_f<frame>.pdb` — the representative structure, for rendering
  in ChimeraX / PyMOL.
- Optionally, PDBs for hand-picked alternative frames.

## 1. Setup

In [ ]:
!pip install -q MDAnalysis

In [ ]:
import os
import pickle
import warnings
from collections import defaultdict

import numpy as np

import MDAnalysis as mda

from google.colab import files, drive

warnings.filterwarnings('ignore')

In [ ]:
drive.mount('/content/drive')

---
## 2. Load the results from `01_hbond_analysis.ipynb`

`PKL_PATH` is the file printed by the last cell of notebook 01.

In [ ]:
# ============== LOAD RESULTS ==============
PKL_PATH = 'INSERT_OUTPUT_DIRECTORY/INSERT_STRUCTURE_NAME_hbond_analysis.pkl'
# ==========================================

with open(PKL_PATH, 'rb') as f:
    all_results = pickle.load(f)

print(f"Loaded: {PKL_PATH}\n")
for name, data in all_results.items():
    cfg = data['config']
    total = sum(r['n_frames'] for r in data['replicates'])
    print(f"{name}")
    print(f"  Replicates: {len(data['replicates'])}, {total} frames total")
    print(f"  Skipped per replicate: {data.get('frames_to_skip', 'not recorded')}"
          f" ({data.get('equilibration_ns', '?')} ns)")
    print(f"  Criteria: <= {cfg.get('hbond_distance')} A, "
          f">= {cfg.get('hbond_angle')} deg | directions: {data.get('directions')}")

---
## 3. Score every frame and select the representative

| Parameter | Meaning |
|---|---|
| `FRAMES_TO_SKIP` | Equilibration frames discarded per replicate. `None` inherits the value used in notebook 01, so the occupancies here match the frequency tables it produced |
| `MIN_FREQ` | A pair is *characteristic* if present in at least this % of production frames |
| `IDENTITY_WEIGHT` | Weight on reproducing the characteristic H-bond network |
| `RMSD_WEIGHT` | Weight on closeness to the average conformation |

In [ ]:
# ============== ADJUST THESE PARAMETERS ==============
# None -> use the same equilibration skip notebook 01 applied (recommended).
FRAMES_TO_SKIP = None

MIN_FREQ = 5.0            # % occupancy for a "characteristic" pair
IDENTITY_WEIGHT = 0.7     # weight on H-bond identity
RMSD_WEIGHT = 0.3         # weight on normalized Ca RMSD
# =====================================================

for name, sys_results in all_results.items():
    print(f"\n{'='*70}\n{name}\n{'='*70}")

    # ---- Resolve the equilibration skip --------------------------------
    if FRAMES_TO_SKIP is None:
        skip = sys_results.get('frames_to_skip')
        if skip is None:
            raise ValueError(
                f"{name}: the results file records no 'frames_to_skip'. "
                "Set FRAMES_TO_SKIP explicitly above."
            )
        print(f"Skipping {skip} frames per replicate (inherited from notebook 01)")
    else:
        skip = FRAMES_TO_SKIP
        recorded = sys_results.get('frames_to_skip')
        print(f"Skipping {skip} frames per replicate (manual override)")
        if recorded is not None and recorded != skip:
            print(f"  Notebook 01 used {recorded}; occupancies below are "
                  "recomputed at the override value.")

    # ---- Pair occupancies over the production frames, pooled -----------
    # Recomputed here so the denominator always matches `skip`.
    analyzed_frames = sum(r['n_frames'] - skip for r in sys_results['replicates'])
    pair_counts = defaultdict(int)
    for rep in sys_results['replicates']:
        for pairs in rep['frame_pairs'][skip:]:
            for p in pairs:
                pair_counts[p] += 1

    frequencies = {p: c / analyzed_frames * 100 for p, c in pair_counts.items()}
    sys_results['frequencies'] = frequencies

    # ---- Characteristic pairs become the scoring weights ---------------
    pair_weights = {p: f for p, f in frequencies.items() if f >= MIN_FREQ}
    print(f"\nCharacteristic pairs (>={MIN_FREQ}% of {analyzed_frames} frames):")
    for (a, b), f in sorted(pair_weights.items(), key=lambda x: -x[1]):
        print(f"  {a} - {b}: {f:.1f}%")

    if not pair_weights:
        print("  (none above threshold - skipping this system)")
        continue

    max_score = sum(pair_weights.values())  # identity denominator

    # ---- Per-frame identity and RMSD -----------------------------------
    frame_data = []
    for rep_idx, rep in enumerate(sys_results['replicates']):
        for frame_idx in range(skip, rep['n_frames']):
            pairs = rep['frame_pairs'][frame_idx]
            identity = sum(pair_weights.get(p, 0) for p in pairs) / max_score
            frame_data.append({
                'rep': rep['name'],
                'rep_idx': rep_idx,
                'frame': frame_idx,
                'identity': identity,
                'rmsd': rep['rmsd_values'][frame_idx],
                'pairs': pairs,
            })

    # ---- Normalize RMSD across the pooled frames and combine ------------
    rmsds = np.array([f['rmsd'] for f in frame_data])
    identities = np.array([f['identity'] for f in frame_data])
    rmsd_norm = (rmsds - rmsds.min()) / (rmsds.max() - rmsds.min() + 1e-9)
    scores = IDENTITY_WEIGHT * (1 - identities) + RMSD_WEIGHT * rmsd_norm

    # ---- Winner (lowest score) -----------------------------------------
    best = frame_data[int(np.argmin(scores))]
    best['frames_skipped'] = skip
    sys_results['rep_frame'] = best

    print(f"\nBEST: {best['rep']} frame {best['frame']}")
    print(f"  Identity: {best['identity']*100:.1f}% | Ca RMSD: {best['rmsd']:.2f} A")

    print("\n  Characteristic pairs present:")
    for (a, b) in sorted(best['pairs'] & set(pair_weights),
                         key=lambda x: -pair_weights[x]):
        print(f"    + {a} - {b} ({pair_weights[(a, b)]:.1f}%)")

    missing = [p for p in pair_weights if p not in best['pairs']]
    if missing:
        print("\n  Characteristic pairs absent from this frame:")
        for (a, b) in sorted(missing, key=lambda x: -pair_weights[x]):
            print(f"    - {a} - {b} ({pair_weights[(a, b)]:.1f}%)")

    # ---- Runners-up ------------------------------------------------------
    print("\n  Alternatives:")
    for i, idx in enumerate(np.argsort(scores)[:4], 1):
        f = frame_data[idx]
        print(f"    {i}. {f['rep']} f{f['frame']}: "
              f"identity {f['identity']*100:.0f}%, RMSD {f['rmsd']:.2f} A")

---
## 4. Write the representative frame to PDB

The universe is rebuilt from the topology and the replicate trajectory recorded
in `config`, so this works in a fresh session where only the pickle is loaded.

In [ ]:
OUTPUT_DIR = 'INSERT_OUTPUT_DIRECTORY/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

for name, sys_results in all_results.items():
    if 'rep_frame' not in sys_results:
        print(f"No representative frame for {name}, skipping...")
        continue

    rep = sys_results['rep_frame']
    pdb_file = sys_results['config']['pdb_file']
    dcd_file = sys_results['config']['dcd_files'][rep['rep_idx']]

    u = mda.Universe(pdb_file, dcd_file)
    u.trajectory[rep['frame']]          # seek to the chosen frame

    pdb_name = f"{OUTPUT_DIR}{name}_rep_{rep['rep']}_f{rep['frame']}.pdb"
    u.atoms.write(pdb_name)

    print(f"{name}: {rep['rep']} frame {rep['frame']} | "
          f"Identity: {rep['identity']*100:.1f}% | Ca RMSD: {rep['rmsd']:.2f} A")
    print(f"  Saved: {pdb_name}")
    files.download(pdb_name)

print("\nDone.")

---
## 5. (Optional) Export hand-picked alternative frames

Fill `alternatives` with entries from the "Alternatives" list printed above —
useful when the top-ranked frame is unsuitable for a figure for a reason the
score cannot capture. `rep_idx` is the 0-based index into `config['dcd_files']`.

In [ ]:
SYSTEM_TO_EXPORT = 'INSERT_STRUCTURE_NAME'   # must match a key of all_results

alternatives = [
    # {'rep_idx': 0, 'rep': 'Rep1', 'frame': 0},
    # {'rep_idx': 2, 'rep': 'Rep3', 'frame': 0},
]

sys_results = all_results[SYSTEM_TO_EXPORT]
pdb_file = sys_results['config']['pdb_file']
dcd_files = sys_results['config']['dcd_files']

for alt in alternatives:
    u = mda.Universe(pdb_file, dcd_files[alt['rep_idx']])
    u.trajectory[alt['frame']]

    pdb_name = f"{OUTPUT_DIR}{SYSTEM_TO_EXPORT}_{alt['rep']}_f{alt['frame']}.pdb"
    u.atoms.write(pdb_name)

    print(f"Saved: {pdb_name}")
    files.download(pdb_name)

print("\nDone.")